# Chaîne RAG : recherche filtrée

**Objectif (partie A)** : vérifier sur l'index complet (Nouvelle-Aquitaine, photo du 29/10/2026) que la recherche filtrée corrige les défauts relevés par les tests de recherche du notebook 03 :
- la ville demandée n'est pas toujours respectée (« théâtre à Poitiers » renvoyait un événement à Châtellerault) ;
- un même événement peut occuper plusieurs des 5 résultats ;
- des événements passés se mêlent aux événements à venir ;
- enfin, constat fait sur les données nettoyées : la plage début–fin d'un événement ne dit pas s'il a lieu un jour donné (un atelier hebdomadaire de septembre à juin couvre tous les week-ends).

**Consignes** (énoncé, étape 4) : « Utilisez LangChain pour orchestrer les appels entre la base Faiss et le modèle Mistral » ; « S'assurer que les réponses générées sont à la fois pertinentes et bien formulées ».

**Fonctionnement** (`rag/criteria.py`, `rag/retrieval.py`)
1. Des règles repèrent dans la question la ville (seulement après une préposition de lieu : « à », « sur », « près de »…), le département et la période (« ce week-end », « en décembre »… ; « à venir » par défaut).
2. Seuls les événements qui respectent ces critères, avec **au moins une séance** dans la période, sont autorisés.
3. La recherche par le sens porte sur **tous** les chunks autorisés : par défaut, le `FAISS` de LangChain ne filtre que les 20 chunks les plus proches.
4. On garde le meilleur chunk de chaque événement, et 5 événements.

**Date du jour fixée au jeudi 29/10/2026**, date de la photo : les résultats sont reproductibles.

**Prérequis** : l'index construit avec `python -m scripts.build_index`, et la clé `MISTRAL_API_KEY` dans `.env`. Chaque question est vectorisée par Mistral (une requête par seconde au maximum).

## A1. Chargement

In [2]:
import sys
import time
from datetime import date, datetime
from pathlib import Path

# Le notebook s'exécute depuis notebooks/ : on ajoute la racine du projet au chemin d'import.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from rag import config
from rag.criteria import Criteria, extract_criteria, normalize
from rag.indexing import load_manifest, load_vectorstore, vectorstore_dir_for
from rag.retrieval import EventRetriever, overlaps, session_dates, sessions_in_period

load_dotenv(PROJECT_ROOT / ".env")  # charge MISTRAL_API_KEY (jamais affichée)
pd.set_option("display.max_colwidth", 60)

TODAY = date(2026, 10, 29)  # jeudi, date de la photo de référence

INDEX_DIR = vectorstore_dir_for(config.REFERENCE_RAW_FILE)
store = load_vectorstore(INDEX_DIR)
manifest = load_manifest(INDEX_DIR)

start = time.perf_counter()
retriever = EventRetriever(store)   # lit les métadonnées une fois : villes, départements, séances
preparation_ms = (time.perf_counter() - start) * 1000

print(f"Index {INDEX_DIR.name} : {manifest['chunks']} chunks, {manifest['events']} événements")
print(f"Préparation des filtres : {preparation_ms:.0f} ms ({len(retriever.events)} événements, "
      f"{len(retriever.city_index)} villes)")

/Users/nadebaque/Documents/Ai Engineer/Projet_7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index nouvelle-aquitaine_2026-10-29 : 20644 chunks, 18771 événements
Préparation des filtres : 94 ms (18771 événements, 1566 villes)


## A2. Critères repérés dans les questions

Aucun appel à Mistral : les critères sont repérés par des règles.

**Hypothèses** :
- une ville précédée d'une préposition de lieu est repérée ; « dégustation de cognac » ne doit pas donner la ville de Cognac ;
- « jazz Bordeaux samedi », sans préposition, ne donne pas de ville : c'est une limite connue, la recherche par le sens prend alors le relais ;
- sans expression de temps, la période est « à venir » (à partir du 29/10/2026) ;
- la question hors sujet ne donne aucun critère.

In [3]:
QUESTIONS = [
    "Un concert de jazz à Bordeaux ce week-end",
    "Du théâtre à Poitiers",
    "Une exposition à Limoges en décembre",
    "Que faire avec des enfants dans les Landes cette semaine ?",
    "Une balade dans la nature demain",
    "Un spectacle à La Rochelle le 14 novembre",
    "Quels concerts ont eu lieu à Pau le mois dernier ?",
    "Une dégustation de cognac",
    "jazz Bordeaux samedi",
    "Quelle est la capitale de l'Australie ?",
]


def describe(criteria: Criteria) -> dict:
    """Critères repérés, en colonnes lisibles."""
    period = criteria.period
    return {
        "ville": criteria.city or "",
        "département": criteria.department or "",
        "période": period.label,
        "du": period.start.isoformat() if period.start else "",
        "au": period.end.isoformat() if period.end else "",
    }


pd.DataFrame([{"question": question, **describe(extract_criteria(question, TODAY, retriever.city_index))}
              for question in QUESTIONS])

,question,ville,département,période,du,au
0,Un concert de jazz à Bordeaux ce week-end,Bordeaux,,ce week-end,2026-10-31,2026-11-01
1,Du théâtre à Poitiers,Poitiers,,à venir,2026-10-29,
2,Une exposition à Limoges en décembre,Limoges,,en décembre 2026,2026-12-01,2026-12-31
3,Que faire avec des enfants dans les Landes cette semaine ?,,Landes,cette semaine,2026-10-29,2026-11-01
4,Une balade dans la nature demain,,,demain,2026-10-30,2026-10-30
5,Un spectacle à La Rochelle le 14 novembre,La Rochelle,,le samedi 14 novembre 2026,2026-11-14,2026-11-14
6,Quels concerts ont eu lieu à Pau le mois dernier ?,Pau,,le mois dernier,2026-09-01,2026-09-30
7,Une dégustation de cognac,,,à venir,2026-10-29,
8,jazz Bordeaux samedi,,,samedi,2026-10-31,2026-10-31
9,Quelle est la capitale de l'Australie ?,,,à venir,2026-10-29,


## A3. Avant / après : recherche brute et recherche filtrée

Pour chaque question : la **recherche brute** (les 5 chunks les plus proches, comme dans le notebook 03), puis la **recherche filtrée** (5 événements). Indicateurs :
- **dans la ville** : résultats situés dans la ville demandée ;
- **événements distincts** : un même événement ne doit compter qu'une fois ;
- **passés** : résultats dont toutes les séances sont terminées à la date du jour ;
- **séance dans la période** : résultats qui ont au moins une séance dans la période demandée.

**Hypothèses** : la recherche filtrée donne 5 résultats sur 5 dans la ville et dans la période, 5 événements distincts et aucun événement passé. Pour la question hors sujet, elle renvoie toujours 5 événements à venir : seul le LLM pourra les écarter.

In [4]:
COMPARED = [
    "Du théâtre à Poitiers",
    "Un concert de jazz à Bordeaux ce week-end",
    "Une exposition à Limoges",
    "Quelle est la capitale de l'Australie ?",
]


def first_session_in_period(sessions: list[dict], criteria: Criteria) -> str:
    """Début de la première séance dans la période demandée, ou « aucune »."""
    matching = sessions_in_period(sessions, criteria.period)
    return min(session["begin"] for session in matching)[:16].replace("T", " ") if matching else "aucune"


def results_table(results: list, criteria: Criteria) -> pd.DataFrame:
    """Une ligne par résultat (document, score)."""
    return pd.DataFrame([{
        "score": round(float(score), 3),
        "titre": document.metadata["title"][:55],
        "ville": document.metadata["city"],
        "1re séance dans la période": first_session_in_period(document.metadata["sessions"], criteria),
        "passé": all(session_dates(session)[1] < TODAY for session in document.metadata["sessions"]),
        "événement": document.metadata["uid"],
    } for document, score in results])


def indicators(table: pd.DataFrame, criteria: Criteria) -> dict:
    """Indicateurs d'une liste de résultats."""
    in_city = (table["ville"].map(normalize) == normalize(criteria.city)).sum()
    return {
        "dans la ville": f"{in_city}/{len(table)}" if criteria.city else "—",
        "événements distincts": f"{table['événement'].nunique()}/{len(table)}",
        "passés": int(table["passé"].sum()),
        "séance dans la période": f"{(table['1re séance dans la période'] != 'aucune').sum()}/{len(table)}",
    }


comparison = []
for question in COMPARED:
    criteria = extract_criteria(question, TODAY, retriever.city_index)
    vector = store.embeddings.embed_query(question)          # requête à Mistral
    time.sleep(1.1)                                          # 1 requête par seconde au maximum
    before = results_table(store.similarity_search_with_score_by_vector(vector, k=5), criteria)
    after = results_table([(found.document, found.score) for found in retriever.search(question, criteria)],
                          criteria)
    time.sleep(1.1)
    print(f"\n« {question} » ; critères : {describe(criteria)}")
    print("Avant (recherche brute, 5 chunks) :")
    display(before)
    print("Après (recherche filtrée, 5 événements) :")
    display(after)
    comparison += [{"question": question, "recherche": "brute", **indicators(before, criteria)},
                   {"question": question, "recherche": "filtrée", **indicators(after, criteria)}]


« Du théâtre à Poitiers » ; critères : {'ville': 'Poitiers', 'département': '', 'période': 'à venir', 'du': '2026-10-29', 'au': ''}
Avant (recherche brute, 5 chunks) :


,score,titre,ville,1re séance dans la période,passé,événement
0,0.841,Visites techniques du TAP,Poitiers,aucune,True,39694355
1,0.839,Visite du TAP,Poitiers,aucune,True,88930307
2,0.827,LIEUX COMMUNS,Poitiers,aucune,True,29726681
3,0.827,Visite en famille du TAP,Poitiers,aucune,True,78138163
4,0.825,Exposition : « L’Odyssée du théâtre municipal de Châtel,Châtellerault,aucune,True,24359274


Après (recherche filtrée, 5 événements) :


,score,titre,ville,1re séance dans la période,passé,événement
0,0.813,Pourquoi les poules préfèrent être élevées en batterie,Poitiers,2027-03-12 20:00,False,98769998
1,0.812,Delete,Poitiers,2027-01-13 21:00,False,27565394
2,0.809,L'heure blanche,Poitiers,2026-12-15 14:00,False,38942905
3,0.804,Les primitifs de la photographie en Poitou au 19e siècl,Poitiers,2026-11-25 18:00,False,61766993
4,0.803,Photographier les grands ensembles : l'exemple des Cour,Poitiers,2027-01-28 18:00,False,90997125



« Un concert de jazz à Bordeaux ce week-end » ; critères : {'ville': 'Bordeaux', 'département': '', 'période': 'ce week-end', 'du': '2026-10-31', 'au': '2026-11-01'}
Avant (recherche brute, 5 chunks) :


,score,titre,ville,1re séance dans la période,passé,événement
0,0.849,Concert Jazz & Co,Bordeaux,aucune,False,97173163
1,0.842,Petit concert de Jazz,Bordeaux,aucune,True,38962553
2,0.838,Concert du dimanche,Bordeaux,aucune,False,84607589
3,0.828,Mardis de la musique : La France et le jazz,Bordeaux,aucune,True,2708786
4,0.822,BIG BAND DU CONSERVATOIRE DE BORDEAUX,Cenon,aucune,True,41263281


... (6 outputs left out)


« Quelle est la capitale de l'Australie ? » ; critères : {'ville': '', 'département': '', 'période': 'à venir', 'du': '2026-10-29', 'au': ''}
Avant (recherche brute, 5 chunks) :


,score,titre,ville,1re séance dans la période,passé,événement
0,0.801,Vins d'Australie,Bordeaux,aucune,True,24304870
1,0.796,Vins d'Australie,Bordeaux,aucune,True,24304870
2,0.790,"Cap sur les vins d'Adelaide, Australie, et de Hawke's B",Bordeaux,aucune,True,27023041
3,0.781,"Bouger à l'étranger, c'est possible ?! SPÉCIAL AUSTRALI",,2026-11-04 18:00,False,46137515
4,0.779,"Webinaire ""Partir en Australie & Nouvelle-Zélande""",,aucune,True,69515437


Après (recherche filtrée, 5 événements) :


,score,titre,ville,1re séance dans la période,passé,événement
0,0.781,"Bouger à l'étranger, c'est possible ?! SPÉCIAL AUSTRALI",,2026-11-04 18:00,False,46137515
1,0.758,BRASILIA - Photographies de Lucien Clergue,Bordeaux,2026-10-29 14:00,False,93754181
2,0.753,La quête d'Halloween,Bordeaux,2026-10-30 10:00,False,13114214
3,0.749,Rencontre avec Karim Kattan,Bordeaux,2026-11-14 18:00,False,3932144
4,0.749,Le concierge vous accueille,Bordeaux,2026-11-07 15:30,False,41759101


In [5]:
# Synthèse des indicateurs, avant et après
pd.DataFrame(comparison)

,question,recherche,dans la ville,événements distincts,passés,séance dans la période
0,Du théâtre à Poitiers,brute,4/5,5/5,5,0/5
1,Du théâtre à Poitiers,filtrée,5/5,5/5,0,5/5
2,Un concert de jazz à Bordeaux ce week-end,brute,4/5,5/5,3,0/5
3,Un concert de jazz à Bordeaux ce week-end,filtrée,5/5,5/5,0,5/5
4,Une exposition à Limoges,brute,5/5,5/5,5,0/5
5,Une exposition à Limoges,filtrée,5/5,5/5,0,5/5
6,Quelle est la capitale de l'Australie ?,brute,—,4/5,4,1/5
7,Quelle est la capitale de l'Australie ?,filtrée,—,5/5,0,5/5


## A4. Séances réelles ou plage de dates

Pour le week-end qui suit la date du jour (samedi 31 octobre et dimanche 1er novembre 2026), on compte les événements retenus par chaque méthode, sans appel à Mistral :
- **plage** : la période début–fin de l'événement chevauche le week-end ;
- **séances** : au moins une séance a lieu pendant le week-end (méthode retenue).

**Hypothèse** : on retrouve les chiffres mesurés sur les données nettoyées, soit 314 événements par la plage et 194 par les séances.

In [6]:
weekend = extract_criteria("ce week-end", TODAY, retriever.city_index).period
first_chunks = {document.metadata["uid"]: document.metadata
                for document in (store.docstore.search(doc_id) for doc_id in store.index_to_docstore_id.values())
                if document.metadata["chunk_index"] == 0}


def range_dates(metadata: dict) -> tuple[date, date]:
    """Dates de début et de fin de l'événement entier."""
    return datetime.fromisoformat(metadata["start"]).date(), datetime.fromisoformat(metadata["end"]).date()


by_range = {uid for uid, metadata in first_chunks.items() if overlaps(weekend, *range_dates(metadata))}
by_sessions = retriever.allowed_uids(Criteria(period=weekend))
wrongly_kept = by_range - by_sessions

print(f"Week-end du {weekend.start:%d/%m} au {weekend.end:%d/%m/%Y}")
print(f"Retenus par la plage : {len(by_range)}")
print(f"Retenus par les séances : {len(by_sessions)}")
print(f"Retenus à tort par la plage : {len(wrongly_kept)} ({len(wrongly_kept) / len(by_range):.0%})")
print(f"Retenus par les séances mais pas par la plage : {len(by_sessions - by_range)}")

Week-end du 31/10 au 01/11/2026
Retenus par la plage : 314
Retenus par les séances : 194
Retenus à tort par la plage : 120 (38%)
Retenus par les séances mais pas par la plage : 0


In [7]:
# Exemples d'événements retenus à tort par la plage : aucune séance pendant le week-end
def sessions_around(sessions: list[dict]) -> tuple[str, str]:
    """Dernière séance avant le week-end et première séance après."""
    before = [session["begin"][:10] for session in sessions if session_dates(session)[1] < weekend.start]
    after = [session["begin"][:10] for session in sessions if session_dates(session)[0] > weekend.end]
    return (max(before) if before else "—"), (min(after) if after else "—")


pd.DataFrame([{
    "titre": first_chunks[uid]["title"][:55],
    "plage": f"{first_chunks[uid]['start'][:10]} → {first_chunks[uid]['end'][:10]}",
    "séances": len(first_chunks[uid]["sessions"]),
    "dernière séance avant": sessions_around(first_chunks[uid]["sessions"])[0],
    "première séance après": sessions_around(first_chunks[uid]["sessions"])[1],
} for uid in sorted(wrongly_kept)[:8]])

,titre,plage,séances,dernière séance avant,première séance après
0,Stage en Anglais pendant les vacances scolaire,2026-02-16 → 2027-02-26,16,2026-02-23,2027-02-15
1,Caché au Creux d'un Matelas,2026-09-18 → 2027-01-30,22,2026-09-26,2027-01-06
2,Jeux de Société de Fred à Beyssenac (4e Dimanche du moi,2026-01-25 → 2026-12-27,12,2026-10-25,2026-11-22
3,Rencontre ton métier au musée,2026-10-08 → 2026-12-10,2,2026-10-08,2026-12-10
4,Oscar et la dame rose,2026-10-22 → 2026-12-29,12,2026-10-27,2026-11-03
5,Atelier Techniques mixtes,2026-06-13 → 2026-11-06,2,2026-06-13,2026-11-06
6,Lecture tout-petits,2025-11-05 → 2026-12-16,30,2026-10-21,2026-11-04
7,Sur les traces de Jean Dupas,2026-06-29 → 2026-11-27,108,2026-10-30,2026-11-02


## A5. Temps de la recherche filtrée

Filtrer **tous** les chunks a un coût : Faiss trie les 20 644 chunks, puis LangChain lit les métadonnées de chacun pour appliquer le filtre. On le mesure sur trois cas :
- **filtre sur une ville** : peu d'événements autorisés ;
- **filtre sur un week-end** ;
- **aucun critère** : tous les événements à venir sont autorisés, c'est le cas le plus lent.

**Méthode** : chaque question est vectorisée une fois par Mistral (temps mesuré) ; la partie locale est mesurée 10 fois (médiane). On compare la recherche filtrée à la recherche brute (5 chunks).

**Hypothèse** : la partie locale reste faible devant la vectorisation de la question (≈ 250 ms d'après le notebook 03).

In [8]:
TIMED = ["Du théâtre à Poitiers", "Un concert ce week-end", "Une sortie en famille"]
total = store.index.ntotal


def median_ms(function, repeat: int = 10) -> float:
    """Durée médiane d'un appel, en millisecondes."""
    durations = []
    for _ in range(repeat):
        start = time.perf_counter()
        function()
        durations.append((time.perf_counter() - start) * 1000)
    return float(np.median(durations))


timings = []
for question in TIMED:
    criteria = extract_criteria(question, TODAY, retriever.city_index)
    start = time.perf_counter()
    vector = store.embeddings.embed_query(question)          # requête à Mistral
    embedding_ms = (time.perf_counter() - start) * 1000
    time.sleep(1.1)
    allowed = retriever.allowed_uids(criteria)
    timings.append({
        "question": question,
        "événements autorisés": len(allowed),
        "vectorisation Mistral (ms)": round(embedding_ms),
        "calcul des autorisés (ms)": round(median_ms(lambda: retriever.allowed_uids(criteria)), 1),
        "recherche filtrée, tous les chunks (ms)": round(median_ms(
            lambda: store.similarity_search_with_score_by_vector(
                vector, k=total, fetch_k=total, filter=lambda metadata: metadata["uid"] in allowed)), 1),
        "recherche brute, 5 chunks (ms)": round(median_ms(
            lambda: store.similarity_search_with_score_by_vector(vector, k=5)), 1),
    })
pd.DataFrame(timings)

,question,événements autorisés,vectorisation Mistral (ms),calcul des autorisés (ms),"recherche filtrée, tous les chunks (ms)","recherche brute, 5 chunks (ms)"
0,Du théâtre à Poitiers,40,244,1.3,22.8,1.2
1,Un concert ce week-end,194,211,8.4,23.1,1.2
2,Une sortie en famille,2697,1816,12.6,25.1,1.4


## A6. Diagnostic du classement : que faut-il vectoriser ?

En A3, pour « Du théâtre à Poitiers », la recherche filtrée classe deux conférences sur la photographie en Poitou devant deux pièces de théâtre (« Portrait hypothétique de deux tueurs en série », « Imbécile »). Le filtre garantit déjà la ville et la période. Dans le vecteur de la question, « à Poitiers » ne départage donc plus les candidats : il peut seulement favoriser les événements qui **parlent** de Poitiers, comme ces conférences, qui citent Poitou ou Poitiers trois à quatre fois.

On compare donc deux classements des événements autorisés :
- avec la **question entière**, comme le fait `EventRetriever` ;
- avec la **question réduite** à ce qu'on cherche, sans le lieu ni la période (« Du théâtre »). Les questions réduites sont écrites à la main ici ; si le gain se confirme, `rag/criteria.py` les produira.

Les questions couvrent une ville peu fournie (Poitiers, 40 événements à venir), une ville très fournie (Bordeaux, 1 217), un département (Charente-Maritime, 84) et une ville avec une période (Bordeaux, ce week-end).

**Pertinence** : un événement est jugé pertinent si son titre ou ses mots-clés contiennent le motif cherché (« théâtre », « expo »…). C'est une approximation reproductible, pas une vérité : un spectacle de marionnettes sans le mot « théâtre » compte comme non pertinent, et des mots-clés trop larges trompent la règle (dans les Landes, les huit événements dont les mots-clés contiennent « randonnée » sont des apéros entre célibataires). Le jeu de test annoté servira de référence pour l'évaluation.

**Indicateurs** :
- **pertinents dans les 5** : ce que le LLM recevra ;
- **rang du 1er pertinent** ;
- **rang médian des pertinents** : plus il est petit, mieux l'ensemble des pertinents est classé.

**Hypothèse** : la question réduite fait remonter les pertinents, surtout là où la ville est très présente dans les textes. 8 requêtes à Mistral (une par texte distinct), espacées de 1,1 s.

In [9]:
import re

# (question, question réduite à ce qu'on cherche, motif de pertinence sur le titre et les mots-clés)
DIAGNOSED = [
    ("Du théâtre à Poitiers", "Du théâtre", r"\btheatr"),
    ("Une exposition à Limoges", "Une exposition", r"\bexpo"),
    ("Un concert de jazz à Bordeaux", "Un concert de jazz", r"\bjazz\b"),
    ("Du théâtre en Charente-Maritime", "Du théâtre", r"\btheatr"),
    ("Une exposition à Bordeaux ce week-end", "Une exposition", r"\bexpo"),
]
vectors = {}  # un seul appel à Mistral par texte distinct


def embed(text: str) -> list[float]:
    """Vecteur d'un texte, demandé à Mistral la première fois seulement."""
    if text not in vectors:
        vectors[text] = store.embeddings.embed_query(text)   # requête à Mistral
        time.sleep(1.1)                                       # 1 requête par seconde au maximum
    return vectors[text]


def is_relevant(metadata: dict, pattern: str) -> bool:
    """Le titre ou les mots-clés de l'événement contiennent le motif (textes normalisés)."""
    return bool(re.search(pattern, normalize(" ".join([metadata["title"], *(metadata["keywords"] or [])]))))


def ranked_events(vector: list[float], allowed: set[str]) -> list[tuple[str, float]]:
    """Tous les événements autorisés, du plus proche au moins proche (score de leur meilleur chunk)."""
    total = store.index.ntotal
    matches = store.similarity_search_with_score_by_vector(
        vector, k=total, fetch_k=total, filter=lambda metadata: metadata["uid"] in allowed)
    best = {}
    for document, score in matches:   # du plus au moins similaire : le premier chunk est le meilleur
        best.setdefault(document.metadata["uid"], float(score))
    return list(best.items())


# first_chunks (A4) : les métadonnées du premier chunk de chaque événement
diagnosis, rankings = [], {}
for question, reduced, pattern in DIAGNOSED:
    criteria = extract_criteria(question, TODAY, retriever.city_index)
    allowed = retriever.allowed_uids(criteria)
    relevant = {uid for uid in allowed if is_relevant(first_chunks[uid], pattern)}
    for variant, text in [("entière", question), ("réduite", reduced)]:
        rankings[question, variant] = ranked_events(embed(text), allowed)
        ranks = [rank for rank, (uid, _) in enumerate(rankings[question, variant], start=1) if uid in relevant]
        diagnosis.append({
            "question": question,
            "texte vectorisé": text,
            "autorisés": len(allowed),
            "pertinents": len(relevant),
            "pertinents dans les 5": sum(rank <= 5 for rank in ranks),
            "rang du 1er pertinent": ranks[0],
            "rang médian des pertinents": float(np.median(ranks)),
        })
pd.DataFrame(diagnosis)

,question,texte vectorisé,autorisés,pertinents,pertinents dans les 5,rang du 1er pertinent,rang médian des pertinents
0,Du théâtre à Poitiers,Du théâtre à Poitiers,40,4,2,1,4.5
1,Du théâtre à Poitiers,Du théâtre,40,4,3,1,4.5
2,Une exposition à Limoges,Une exposition à Limoges,99,10,3,1,10.0
3,Une exposition à Limoges,Une exposition,99,10,4,1,8.0
4,Un concert de jazz à Bordeaux,Un concert de jazz à Bordeaux,1217,12,4,1,10.5
5,Un concert de jazz à Bordeaux,Un concert de jazz,1217,12,3,1,15.0
6,Du théâtre en Charente-Maritime,Du théâtre en Charente-Maritime,84,5,3,1,5.0
7,Du théâtre en Charente-Maritime,Du théâtre,84,5,1,2,10.0
8,Une exposition à Bordeaux ce week-end,Une exposition à Bordeaux ce week-end,103,36,4,2,19.5
9,Une exposition à Bordeaux ce week-end,Une exposition,103,36,5,1,18.5


In [10]:
def top_five(question: str, variant: str, pattern: str) -> list[str]:
    """Les 5 premiers événements d'un classement avec leur score ; « * » : pertinent selon la règle."""
    return [f"{'* ' if is_relevant(first_chunks[uid], pattern) else ''}{first_chunks[uid]['title'][:45]} ({score:.3f})"
            for uid, score in rankings[question, variant][:5]]


for question, reduced, pattern in DIAGNOSED:
    print(f"\n« {question} »")
    display(pd.DataFrame({"question entière": top_five(question, "entière", pattern),
                          f"« {reduced} »": top_five(question, "réduite", pattern)}, index=range(1, 6)))


« Du théâtre à Poitiers »


,question entière,« Du théâtre »
1,* Pourquoi les poules préfèrent être élevées en (0.813),* Delete (0.779)
2,* Delete (0.812),L'heure blanche (0.768)
3,L'heure blanche (0.809),LA TÊTE MI-PNEU MI-PUNK (0.759)
4,Les primitifs de la photographie en Poitou au (0.804),* Portrait hypothétique de deux tueurs en série (0.752)
5,Photographier les grands ensembles : l'exempl (0.803),* Imbécile (0.751)



« Une exposition à Limoges »


,question entière,« Une exposition »
1,* Exposition Écrits du Limousin en portraits d' (0.845),* Recto/Verso exposition vol.1 (0.809)
2,"""Sels de terre et grains d'argent. Céramique (0.836)",* SANS CONTREFAÇON — Anatomie d'un vêtement : d (0.797)
3,""" C'est bientôt Noël "" (0.823)","""Sels de terre et grains d'argent. Céramique (0.793)"
4,"* Exposition photo : La ville invisible, Guilla (0.820)","* Exposition photo : La ville invisible, Guilla (0.792)"
5,* Recto/Verso exposition vol.1 (0.818),* Objectif Rose - exposition photographique (0.792)



« Un concert de jazz à Bordeaux »


,question entière,« Un concert de jazz »
1,* Concert Jazz & Co (0.870),* Concert Jazz & Co (0.878)
2,Concert du dimanche (0.851),Concert du dimanche (0.853)
3,* Belmondo X Belmondo (0.833),Concert de l'ensemble vocal atout chœur du Ta (0.836)
4,* Brad Mehldau (0.826),* Belmondo X Belmondo (0.831)
5,* Freedom Jazz Dance (0.825),* Moonlight Sessions #2 (0.829)



« Du théâtre en Charente-Maritime »


,question entière,« Du théâtre »
1,* M'aime Pas Mort (0.772),◯ ~ Ode à la douceur (0.762)
2,Bestiaire Vestiaire (0.754),* M'aime Pas Mort (0.759)
3,◯ ~ Ode à la douceur (0.747),Paroles d'Ombres (0.755)
4,* Imbécile (0.744),Les Diables (0.754)
5,* Imbécile (0.741),Les Enfants du siècle (0.753)



« Une exposition à Bordeaux ce week-end »


,question entière,« Une exposition »
1,10 pour DYS : L'Odyssée des invisibles (0.801),* Mine d'Emile s'expose (0.820)
2,* Exposition Jean Dupas & Co. Le grand Art déco (0.800),* Exposition Sols (0.819)
3,* Exposition Miaou! (0.797),"* Exposition ""Un globe, des mondes"" (0.817)"
4,* Mine d'Emile s'expose (0.796),* Un habit trop grand (0.808)
5,* Exposition Sols (0.796),"* Exposition ""Céramiques, corps sensibles"" (0.805)"


## A7. Synthèse de la partie A

**Les quatre défauts relevés dans le notebook 03 sont corrigés** (A3, 4 questions) :

| Indicateur | Recherche brute | Recherche filtrée |
|---|---|---|
| Résultats dans la ville demandée | 4/5 pour deux questions sur trois | 5/5 |
| Événements distincts | 4/5 pour une question (2 chunks du même événement) | 5/5 |
| Événements passés | 3 à 5 sur 5 | 0 |
| Résultats avec une séance dans la période | 0 ou 1 sur 5 | 5/5 |

- **Critères (A2)** : repérés comme attendu sur les 10 questions, y compris les cas limites (« dégustation de cognac » ne donne pas de ville ; « jazz Bordeaux samedi », sans préposition, non plus ; la question hors sujet ne donne aucun critère).
- **Séances (A4)** : pour le week-end du 31/10, la plage début–fin retient 314 événements, les séances 194. Ainsi, 120 événements (38 %) n'ont aucune séance ce week-end ; aucun n'est perdu dans l'autre sens.
- **Coût (A5)** : filtrer tous les chunks prend environ 23 ms (contre environ 1 ms pour 5 chunks), quel que soit le nombre d'événements autorisés. La vectorisation de la question par Mistral prend de 200 à 1 100 ms : le surcoût est négligeable.

**Ce que les filtres ne règlent pas : la pertinence.**
- **Les données peuvent ne pas contenir la réponse.** Parmi les 103 événements à Bordeaux le week-end du 31/10, aucun ne parle de jazz ; le seul concert (musique de chambre) est classé premier. C'est au LLM de le dire, et non de présenter les autres résultats comme du jazz. Même chose pour la question hors sujet.
- **Le classement par le sens est imprécis à l'intérieur des candidats.** Les scores sont serrés (de 0,75 à 0,85) et le modèle rapproche des genres voisins : théâtre, cirque, cinéma, conférence. Vectoriser la question sans le lieu ni la période (A6) corrige le cas de Poitiers (les conférences sur le Poitou disparaissent), mais en dégrade un autre (en Charente-Maritime, des films remontent). Au total, on compte 16 pertinents sur 25 dans les deux cas. **On garde donc la question entière.**
- **Les données sont inégalement réparties** : 1 217 événements à venir à Bordeaux, 84 dans toute la Charente-Maritime, 31 dans les Landes.

**Suite** : les pistes d'amélioration du classement (question réduite, recherche hybride par mots et par le sens, reclassement par un second modèle) seront comparées sur le jeu de test annoté. Elles seront départagées par un jugement humain de la pertinence et des métriques d'évaluation (Ragas), et non par une règle sur les mots, qui se trompe dans les deux sens (des apéros comptés comme randonnées, des expositions sans le mot « exposition »).